# Prompt Complexity Classifier & Arena English Prompts

This notebook demonstrates:
1. **Arena English Prompts**: Load the included real user prompts and preference metadata.
2. **Prompt Classifier**: Score prompts across 14 parameters and route to **Nano**, **Super**, or **Ultra** tiers. The bundled model uses synthetic training labels.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

# Ensure project root is in sys.path
PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project Root:", PROJECT_ROOT)

## 1. Load Arena English Prompts
Load `data/arena/arena_prompts_english.parquet`. Routing and complexity fields are placeholders for future annotation.

In [ ]:
df_arena = pd.read_parquet(PROJECT_ROOT / "data/arena/arena_prompts_english.parquet")
print(f"Loaded Arena: {len(df_arena):,} rows x {len(df_arena.columns)} columns")
df_arena[["prompt_id", "prompt", "model_a", "model_b", "human_preference"]].head()

## 2. Test Prompt Classifier on Real English Prompts
Score prompts across 14 parameters and route to **Nano**, **Super**, or **Ultra** models.

In [ ]:
from tools.prompt_classifier import classify_prompt

sample_prompts = df_arena["prompt"].dropna().head(3).tolist()

for prompt in sample_prompts:
    res = classify_prompt(prompt)
    print("=" * 75)
    print(f"PROMPT:            {prompt}")
    print(f"Route Tier:        {res.route_tier.upper()}")
    print(f"Recommended Model: {res.recommended_model}")
    print(f"Weighted Score:    {res.weighted_score:.3f}")
    print(f"Task Type:         {res.scores.predicted_task_type}")

## 3. Interactive Custom Prompt & 14-Parameter Breakdown
Input your own prompt to see the complete rubric breakdown.

In [ ]:
my_prompt = "Design an automated order routing microservice in Python with asyncio, redis pub/sub, and circuit breaker retry logic."

res = classify_prompt(my_prompt)
print(f"Result: {res.route_tier.upper()} -> {res.recommended_model} (Weighted Score: {res.weighted_score:.3f})")

# Display parameter score table
scores_dict = res.to_dict()["scores"]
scores_df = pd.DataFrame(list(scores_dict.items()), columns=["Rubric Parameter", "Score"])
scores_df